In [1]:
from datetime import datetime, date
import pandas as pd
import itertools
from finrl.agents.elegantrl.models import DRLAgent
from enum import StrEnum, auto
import numpy as np
from torch import optim
import gymnasium as gym
import tempfile
import ray
from ray import train
from ray.train import Checkpoint
from ray import tune
from ray.air import session
from ray.tune import CLIReporter
from ray.tune.schedulers import ASHAScheduler
from finrl.meta.data_processor import DataProcessor
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from stable_baselines3.common.logger import configure
from finrl.meta.env_stock_trading.env_forex_price_trailing import ForexPriceTrailingEnv
import torch as th
import torch
import torch.nn as nn
from collections import deque
import random
import os
import torch.nn.functional as F
from elegantrl.train.config import Config
from elegantrl.train.run    import train_agent
import torch.nn as nns
from elegantrl.agents import AgentDQN, AgentDoubleDQN
from copy import deepcopy
%matplotlib inline

In [2]:
class Currency(StrEnum):
    USD = auto()
    EUR = auto()
    JPY = auto()
    GBP = auto()
    AUD = auto()
    CAD = auto()
    CHF = auto()
    NZD = auto()
    CNY = auto()

In [3]:
start_date: date = date(2010, 1, 1)
end_date: date = date(2020, 1, 1)

majors = [
    "EURUSD=X","USDJPY=X","GBPUSD=X",
    "AUDUSD=X","USDCAD=X","USDCHF=X","NZDUSD=X"
]

# 2) Top non-USD crosses
crosses = [
    "EURGBP=X","EURJPY=X","GBPJPY=X","AUDJPY=X",
    "CADJPY=X","EURAUD=X","EURCAD=X","EURCHF=X",
    "GBPCHF=X","AUDCAD=X","NZDJPY=X","NZDCAD=X"
]

# 3) Key CNY pairs
cny_pairs = [
    "USDCNY=X","EURCNY=X","JPY CNY=X".replace(" ",""),  # -> "JPYCNY=X"
    "GBPCNY=X","AUDCNY=X","CADCNY=X","CHFCNY=X","NZDCNY=X"
]

# 4) Stitch together, then take the first 20 unique
all_tickers = majors + crosses + cny_pairs
# remove any duplicates and slice to 20
seen = set()
forex_ticks: tuple[str, ...] = tuple(
    t for t in all_tickers
    if not (t in seen or seen.add(t))
)
len(forex_ticks)

27

In [4]:
yfd = YahooDownloader(start_date=str(start_date), end_date=str(end_date), ticker_list=forex_ticks)
df = yfd.fetch_data()
df

YF deprecation warning: set proxy via new config function: yf.set_config(proxy=proxy)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Shape of DataFrame:  (70305, 8)


Price,date,close,high,low,open,volume,tic,day
0,2010-01-01,0.945120,0.945720,0.944000,0.945510,0,AUDCAD=X,4
1,2010-01-01,6.009700,6.037000,6.003100,6.036400,0,AUDCNY=X,4
2,2010-01-01,83.449997,83.602997,83.202003,83.463997,0,AUDJPY=X,4
3,2010-01-01,0.898473,0.898473,0.897827,0.898311,0,AUDUSD=X,4
4,2010-01-01,6.397100,6.397300,6.386700,6.386700,0,CADCNY=X,4
...,...,...,...,...,...,...,...,...
70300,2019-12-31,0.673478,0.675621,0.672088,0.673559,0,NZDUSD=X,1
70301,2019-12-31,1.306060,1.306080,1.295310,1.305800,0,USDCAD=X,1
70302,2019-12-31,0.968630,0.969680,0.964500,0.968640,0,USDCHF=X,1
70303,2019-12-31,6.985700,6.985900,6.957500,6.985700,0,USDCNY=X,1


In [5]:
def add_fx_features_for_tick(g: pd.DataFrame) -> pd.DataFrame:
    g["close_prev"] = g.close.shift(1)
    g["high_prev"] = g.high.shift(1)
    g["low_prev"] = g.low.shift(1)

    g["x1"] = (g.close - g.close_prev) / g.close_prev
    g["x2"] = (g.high - g.high_prev) / g.high_prev
    g["x3"] = (g.low - g.low_prev) / g.low_prev
    g["x4"] = (g.high - g.close) / g.close
    g["x5"] = (g.close - g.low) / g.close
    return g

def add_fx_features(df: pd.DataFrame, tic_col: str = "tic") -> pd.DataFrame:
    df_with_features = (
        df
        .groupby(tic_col, group_keys=False)
        .apply(add_fx_features_for_tick)
        .drop(columns=["close_prev", "high_prev", "low_prev"])
        .fillna(0)
    )
    return df_with_features

In [6]:
df_features = add_fx_features(df)
df_features

/var/folders/23/n0prkghs6v94ssbsv_9xq74c0000gn/T/ipykernel_71572/2277383529.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_fx_features_for_tick)


Price,date,close,high,low,open,volume,tic,day,x1,x2,x3,x4,x5
0,2010-01-01,0.945120,0.945720,0.944000,0.945510,0,AUDCAD=X,4,0.000000,0.000000,0.000000,0.000635,0.001185
1,2010-01-01,6.009700,6.037000,6.003100,6.036400,0,AUDCNY=X,4,0.000000,0.000000,0.000000,0.004543,0.001098
2,2010-01-01,83.449997,83.602997,83.202003,83.463997,0,AUDJPY=X,4,0.000000,0.000000,0.000000,0.001833,0.002972
3,2010-01-01,0.898473,0.898473,0.897827,0.898311,0,AUDUSD=X,4,0.000000,0.000000,0.000000,0.000000,0.000718
4,2010-01-01,6.397100,6.397300,6.386700,6.386700,0,CADCNY=X,4,0.000000,0.000000,0.000000,0.000031,0.001626
...,...,...,...,...,...,...,...,...,...,...,...,...,...
70300,2019-12-31,0.673478,0.675621,0.672088,0.673559,0,NZDUSD=X,1,0.004229,0.002703,0.002204,0.003182,0.002063
70301,2019-12-31,1.306060,1.306080,1.295310,1.305800,0,USDCAD=X,1,-0.000979,-0.001743,-0.007653,0.000015,0.008231
70302,2019-12-31,0.968630,0.969680,0.964500,0.968640,0,USDCHF=X,1,-0.005217,-0.004435,-0.003101,0.001084,0.004264
70303,2019-12-31,6.985700,6.985900,6.957500,6.985700,0,USDCNY=X,1,-0.001301,-0.001272,-0.002766,0.000029,0.004037


In [7]:
class LSTM_QNet(nn.Module):
    def __init__(self,
                 window:      int = 16,
                 feature_dim: int = 5,
                 lstm_hidden: int = 32,
                 fc_hidden:   int = 64,
                 action_dim:  int = 3):
        super().__init__()
        self.window      = window
        self.feature_dim = feature_dim
        self.action_dim  = action_dim

        # LSTM over (window × feature_dim)
        self.lstm = nn.LSTM(input_size=feature_dim,
                            hidden_size=lstm_hidden,
                            batch_first=True)
        # small FC on top of LSTM
        self.fc_after_lstm = nn.Linear(lstm_hidden, lstm_hidden)
        # two FC layers after concatenating prev-pos one-hot
        self.fc1 = nn.Linear(lstm_hidden + action_dim, fc_hidden)
        self.fc2 = nn.Linear(fc_hidden,       fc_hidden)
        # final head: Q-values for each action
        self.q_head = nn.Linear(fc_hidden, action_dim)

    def get_q_value(self, state: torch.Tensor) -> torch.Tensor:
        """
        state: (B, window*feature_dim + 1)
        returns Q(s,·): (B, action_dim)
        """
        B = state.size(0)
        hist = state[:, : self.window*self.feature_dim]
        prev = state[:, -1].long()               # δ ∈ {-1,0,1}
        seq  = hist.view(B, self.window, self.feature_dim)

        lstm_out, _ = self.lstm(seq)             # (B, window, lstm_hidden)
        h_T         = lstm_out[:, -1, :]         # (B, lstm_hidden)
        h           = F.relu(self.fc_after_lstm(h_T))

        oh          = F.one_hot(prev+1, self.action_dim).float()  # (B,3)
        z           = torch.cat([h, oh], dim=1)                  # (B, lstm+3)
        z1          = F.relu(self.fc1(z))
        z2          = F.relu(self.fc2(z1))
        q           = self.q_head(z2)                             # (B,3)
        return q

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        """
        called during inference: returns greedy action as (B,1)
        """
        return self.get_q_value(state).argmax(dim=1, keepdim=True)

# ─── 3) REPLAY BUFFER ───────────────────────────────────────────────────────────

class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer   = deque(maxlen=capacity)

    def push(self, s, a, r, s2, done):
        self.buffer.append((s, a, r, s2, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s,a,r,s2,d = map(np.stack, zip(*batch))
        return (
            torch.FloatTensor(s),
            torch.LongTensor(a),
            torch.FloatTensor(r),
            torch.FloatTensor(s2),
            torch.FloatTensor(1 - d)
        )

    def __len__(self):
        return len(self.buffer)

# ─── 4) AGENT + TRAIN LOOP ─────────────────────────────────────────────────────

def train_ddqn(
    env: gym.Env,
    q_net:   nn.Module,
    target_q:nn.Module,
    buffer:  ReplayBuffer,
    episodes:       int       = 700,
    max_steps:      int       = 2000,
    batch_size:     int       = 128,
    gamma:          float     = 0.995,
    lr:             float     = 1e-4,
    eps_start:      float     = 1.0,
    eps_end:        float     = 0.01,
    eps_decay:      float     = 300,
    target_update:  int       = 10
):
    optim_q = optim.Adam(q_net.parameters(), lr=lr)
    steps_done = 0
    episode_rewards = []

    for ep in range(1, episodes+1):
        state, _ = env.reset()
        ep_reward = 0.0

        eps = eps_end + (eps_start - eps_end) * \
              np.exp(-1. * steps_done / eps_decay)

        for t in range(max_steps):
            steps_done += 1
            state_v = torch.FloatTensor(state).unsqueeze(0)

            # ε-greedy action
            if random.random() > eps:
                with torch.no_grad():
                    action = q_net(state_v).item()
            else:
                action = env.action_space.sample()

            next_state, reward, done, trunc, _ = env.step(action)
            ep_reward += reward

            buffer.push(state, action, reward, next_state, done or trunc)
            state = next_state

            # learn once we have enough data
            if len(buffer) >= batch_size:
                s_batch, a_batch, r_batch, s2_batch, not_done = buffer.sample(batch_size)

                # current Q
                q_vals = q_net.get_q_value(s_batch)           # (B,3)
                q_a    = q_vals.gather(1, a_batch.unsqueeze(1))  # (B,1)

                # Double-DQN target
                with torch.no_grad():
                    next_q = q_net.get_q_value(s2_batch)         # online net for action selection
                    next_a = next_q.argmax(dim=1, keepdim=True)  # (B,1)
                    q2     = target_q.get_q_value(s2_batch)      # target net for value
                    q2_a   = q2.gather(1, next_a)                # (B,1)
                    q_target = r_batch.unsqueeze(1) + gamma * not_done.unsqueeze(1) * q2_a

                loss = F.smooth_l1_loss(q_a, q_target)

                optim_q.zero_grad()
                loss.backward()
                optim_q.step()

            if done or trunc:
                break
        
        episode_rewards.append(ep_reward)
        # soft update target network
        if ep % target_update == 0:
            target_q.load_state_dict(q_net.state_dict())

        print(f"Episode {ep:03d} | Reward {ep_reward:8.2f} | Eps {eps:.3f}")

    return q_net, target_q, episode_rewards

In [8]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

class LivePlotForexEnv(ForexPriceTrailingEnv):
    """
    Same as before, but only re-plot once every `plot_every` steps.
    """
    def __init__(self, *args, live_plot: bool = False, plot_every: int = 100, **kwargs):
        super().__init__(*args, **kwargs)
        self.live_plot   = live_plot
        self.plot_every  = plot_every
        self._step_count = 0

        # rolling histories
        self._hc, self._hu, self._hl, self._hp = [], [], [], []
        self._hr = []

    def reset(self, *args, **kwargs):
        obs, info = super().reset(*args, **kwargs)
        self._step_count = 0
        # clear histories
        for buf in (self._hc, self._hu, self._hl, self._hp, self._hr):
            buf.clear()
        return obs, info

    def step(self, action: int):
        obs, reward, done, truncated, info = super().step(action)
        self._step_count += 1

        # record at bar t (ptr-1)
        i = self.ptr - 1
        close = float(self.df.loc[i, "close"])
        upper = close * (1 + self.M)
        lower = close * (1 - self.M)
        agent = float(self.agent_price)

        self._hc.append(close)
        self._hu.append(upper)
        self._hl.append(lower)
        self._hp.append(agent)
        self._hr.append(reward)

        if self.live_plot and (self._step_count % self.plot_every == 0 or done or truncated):
            clear_output(wait=True)
            fig, (ax_price, ax_reward) = plt.subplots(2, 1, figsize=(10, 6),
                                                      sharex=True,
                                                      gridspec_kw={"height_ratios":[3,1]})
            # prices + bounds
            ax_price.plot(self._hc,      label="Close")
            ax_price.plot(self._hp, "--", label="Agent")
            ax_price.plot(self._hu, label="Upper", color='gray', alpha=0.5)
            ax_price.plot(self._hl, label="Lower", color='gray', alpha=0.5)
            ax_price.set_ylabel("Price")
            ax_price.legend(loc="upper left")

            # reward
            ax_reward.plot(self._hr, label="Reward", color="tab:green")
            ax_reward.set_ylabel("Reward")
            ax_reward.set_xlabel("Step")
            ax_reward.legend(loc="upper left")

            plt.tight_layout()
            plt.show()

        return obs, reward, done, truncated, info

In [9]:
# # env = LivePlotForexEnv(
# #     full_df=df_features,
# #     window=16,
# #     margin=0.02,
# #     step_frac=0.1,
# #     episode_len=2000,
# #     live_plot=False,
# #     plot_every=20
# # )
# env = ForexPriceTrailingEnv(
#     full_df=df_features,
#     window=16,
#     margin=0.02,
#     step_frac=0.1,
#     episode_len=2000,
# )

# q_net     = LSTM_QNet(window=16, feature_dim=5, lstm_hidden=128, fc_hidden=64, action_dim=3)
# target_q  = LSTM_QNet(window=16, feature_dim=5, lstm_hidden=128, fc_hidden=64, action_dim=3)
# target_q.load_state_dict(q_net.state_dict())

# buf = ReplayBuffer(capacity=100_000)

# trained_q, trained_target = train_ddqn(
#     env=env,
#     q_net=q_net,
#     target_q=target_q,
#     buffer=buf,
#     episodes=700,
#     max_steps=2000,
#     batch_size=512,
#     gamma=0.995,
#     lr=2e-4,
#     eps_start=1.0,
#     eps_end=0.01,
#     eps_decay=50_000,
#     target_update=20,
# )

# # save your model
# torch.save(trained_q.state_dict(), "lstm_ddqn_qnet.pth")

## Fine Tuning with Ray Tune

In [10]:
def tune_train(config, checkpoint_dir=None):
    # Environment initialization
    env = LivePlotForexEnv(
        full_df=df_features,
        window=16,
        margin=0.02,
        step_frac=0.1,
        episode_len=2000,
        live_plot=False
    )   
    # Model intitialization
    q_net = LSTM_QNet(
        window=16,
        feature_dim=5,
        lstm_hidden=config["lstm_hidden"],
        fc_hidden=config["fc_hidden"],
        action_dim=3
    ).to(config["device"])
    target_q = LSTM_QNet(
        window=16,
        feature_dim=5,
        lstm_hidden=config["lstm_hidden"],
        fc_hidden=config["fc_hidden"],
        action_dim=3
    ).to(config["device"])
    target_q.load_state_dict(q_net.state_dict())

    buf = ReplayBuffer(capacity=100_000)

    _, _, rewards = train_ddqn(
        env=env,
        q_net=q_net,
        target_q=target_q,
        buffer=buf,
        episodes=config["episodes"],
        max_steps=config["max_steps"],
        batch_size=config["batch_size"],
        gamma=config["gamma"],
        lr=config["lr"],
        eps_start=1.0,
        eps_end=0.01,
        eps_decay=config["eps_decay"],
        target_update=config["target_update"]
    )  

    avg_reward = np.mean(rewards)
    metrics = {"avg_reward": avg_reward}
    session.report(metrics)

    with tempfile.TemporaryDirectory() as tmpdir:
        print("STORING CHECKPOINT IN: ", tmpdir)
        # Only rank 0 (in DDP) needs to checkpoint
        if train.get_context().get_world_rank() == 0:
            ckpt_data = {
                "episode":   ep,
                "q_net":     q_net.state_dict(),
                "target_q":  target_q.state_dict(),
                "buffer":    list(buf.buffer),  # deque→list
            }
            torch.save(ckpt_data, os.path.join(tmpdir, "model.pt"))
            ckpt = Checkpoint.from_directory(tmpdir)
        else:
            ckpt = None

        train.report(metrics, checkpoint=ckpt)

def tune_ddqn_forex_price_trailing():
    ray.init(ignore_reinit_error=True)
    
    scheduler = ASHAScheduler(
        metric="avg_reward",
        mode="max",
        max_t=100,
        grace_period=5,
        reduction_factor=2
    )
    reporter = CLIReporter(
        metric_columns=["avg_reward", "training_iteration"]
    )

    # hyperparameter search space
    config = {
        "episodes":      50,
        "max_steps":     2000,
        "batch_size":    tune.choice([128, 256, 512]),
        "gamma":         tune.uniform(0.9, 0.999),
        "lr":            tune.loguniform(1e-5, 1e-3),
        "eps_decay":     tune.choice([10_000, 50_000, 100_000]),
        "target_update": tune.choice([10, 20, 50]),
        "lstm_hidden":   tune.choice([32, 64, 128]),
        "fc_hidden":     tune.choice([32, 64, 128]),
        "device":        "cuda" if torch.cuda.is_available() else "cpu"
    }

    analysis = tune.run(
        tune_train,
        resources_per_trial={"cpu": 4},
        config=config,
        num_samples=12,
        scheduler=scheduler,
        progress_reporter=reporter,
        storage_path="/Users/kpetridis/Documents/workplace/FinRL/ray_tune_results",
        name="ddqn_lstm_tuning",
    )

    # pick best trial
    best_trial = analysis.get_best_trial("avg_reward", "max", "last")
    print("Best hyperparameters found were:", best_trial.config)

    # load best checkpoint and save final model weights
    best_checkpoint = analysis.get_best_checkpoint(
        best_trial, metric="avg_reward", mode="max"
    )
    if best_checkpoint:
        print("Best checkpoint path:", best_checkpoint.to_directory("/Users/kpetridis/Documents/workplace/FinRL/ray_tune_results/ddqn_lstm_tuning"))
        checkpoint = torch.load(os.path.join(best_checkpoint.to_directory("/Users/kpetridis/Documents/workplace/FinRL/ray_tune_results/ddqn_lstm_tuning"), "checkpoint.pth"))
        final_q = LSTM_QNet(
            window=16, feature_dim=5,
            lstm_hidden=best_trial.config["lstm_hidden"],
            fc_hidden=best_trial.config["fc_hidden"],
            action_dim=3
        )
        final_q.load_state_dict(checkpoint["q_net"])
        torch.save(final_q.state_dict(), "best_ddqn_lstm_qnet.pth")
        print("Saved best model to best_ddqn_lstm_qnet.pth")

In [ ]:
tune_ddqn_forex_price_trailing()

2025-06-24 15:33:49,015	INFO worker.py:1908 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
2025-06-24 15:33:49,533	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


(raylet) Warning: The actor ImplicitFunc is very large (14 MiB). Check that its definition is not implicitly capturing a large array or other object in scope. Tip: use ray.put() to put large objects in the Ray object store.
== Status ==
Current time: 2025-06-24 15:33:50 (running for 00:00:00.88)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 80.000: None | Iter 40.000: None | Iter 20.000: None | Iter 10.000: None | Iter 5.000: None
Logical resource usage: 8.0/10 CPUs, 0/0 GPUs
Result logdir: /tmp/ray/session_2025-06-24_15-33-47_507280_71572/artifacts/2025-06-24_15-33-49/ddqn_lstm_tuning/driver_artifacts
Number of trials: 12/12 (12 PENDING)
+------------------------+----------+-------+--------------+-------------+-------------+----------+-------------+---------------+-----------------+
| Trial name             | status   | loc   |   batch_size |   eps_decay |   fc_hidden |    gamma |          lr |   lstm_hidden |   target_update |
|------------------------+----------+-------+--------